# Nearest Neighbor Models

<img src="https://s44783.pcdn.co/in/wp-content/uploads/sites/3/2023/03/types-of-machine-learning.jpg.webp">

# Course Outline

1. Introduction to Nearest Neighbor Methods
2. K-Nearest Neighbor (KNN) Algorithm
3. KNN Classification
4. KNN Regression
5. Final Takeaways

# Nearest Neighbor Algorithm

The principle behind nearest neighbor methods is to find a predefined number of training samples closest in distance to the new point, and predict the label from these. The number of samples can be a user-defined constant (k-nearest neighbor learning), or vary based on the local density of points (radius-based neighbor learning). The distance can, in general, be any metric measure: standard Euclidean distance is the most common choice. Neighbors-based methods are known as non-generalizing machine learning methods, since they simply “remember” all of its training data.

`sklearn.neighbors` provides functionality for unsupervised and supervised neighbors-based learning methods. Supervised neighbors-based learning comes in two flavors: **classification** for data with discrete labels, and **regression** for data with continuous labels.

# k-Nearest Neighbor (KNN) Algorithm

K-Nearest Neighbor (KNN) is a supervised learning algorithm used for both regression and classification.

KNN algorithm assumes the similarity between the new data point and the available data points and put this new data point into the category that is the most similar to the available categories.

**How Does It Work?**

KNN algorithm stores all the available data and classifies a new data point based on the similarity. This means when new a data point appears, it will be easily classified into a well suite category. For example, imagine that you wanted to buy a new phone, but you wanted to check first with your friends what are the brands they are using so you can decide. Then, you buy the most bought brand as your most of your friends. That is KNN.

KNN tries to predict the category for the new data point by calculating the distance between the data point and all the training data points. Then select the K number of points which is closet to the data point. The KNN algorithm will then assign the most category found from the K number of points selected.

Model output will be the the **mode** (if classification) or **mean** (if regression) of the k-nearest neighbor’s target variable.

## KNN Classification

Let’s say you have this dataset that consists of 2 features and 2 categories (Squares and Rectangles), and you want to classify a new data point to one of the 2 categories available.

<fig>
    <img src="https://mlarchive.com/wp-content/uploads/2022/09/img2.png">
    <figcaption>KNN example with 2 Categories and 2 Features</figcaption>
</fig>

The next step is to choose the K value, the K stands for the number of neighboring samples you want to check their labels / categories.

For example K = 3, we will look at the closest 3 data points, this can be done using the Euclidian distance.

Euclidean distance gives the distance between two points or the straight line distance. Assume that (x1,y1) and (x2,y2) are two points, then the Euclidean distance can then be calculated by the formula:

$$ \text{d} = \sqrt{(x2-x1)^2 + (y2-y1)^2} $$

By looking at the minimum distances computed, we define our neighbors.

<fig>
    <img src="https://mlarchive.com/wp-content/uploads/2022/09/img1-1.png">
    <figcaption>KNN example with K=3. The Red circle illustrates the closest K points to the new data point, when K=3</figcaption>
</fig>

Because the 3 closest points are 2 Triangles and 1 Square, therefore, we choose the most voted which is a Traiangle category.

For this section we will use the iris dataset to demonstrate the machine learning workflow for KNN.

<img src="https://s3.amazonaws.com/assets.datacamp.com/blog_assets/Machine+Learning+R/iris-machinelearning.png">

### Example

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

SEED = 42

#### Load Dataset

In [ ]:
df_clf = load_iris(as_frame=True).frame

df_clf.head()

#### Inspect Dataset

In [ ]:
df_clf.info()

**NOTE: Setting up a Baseline using the Proportional Chance Criterion (PCC)**

PCC represents the probability that a classifier will correctly classify an instance by chance, based on the distribution of classes in the dataset. It is calculated as:

$$ \text{PCC} = \sum_{i=1}^{n} p_i^2 $$

Where $p_i$ is the proportion of instances belonging to class $i$ in the dataset, and $n$ is the total number of classes. The PCC provides a baseline accuracy that any classifier should exceed to be considered effective. If a model's accuracy is close to or below the PCC, it may indicate that the model is not learning effectively from the data and is performing no better than random guessing.

A good baseline is 1.25 * PCC


In [ ]:
# PCC Calculation
class_counts = df_clf["target"].value_counts()
class_proportions = class_counts / len(df_clf)
pcc = (class_proportions**2).sum()
print(f"Proportional Chance Criterion (PCC): {1.25 * pcc:.4f}")

#### Train-Test Split (75-25 split)

In [ ]:
X = df_clf.drop("target", axis=1)
y = df_clf["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

#### Exploratory Data Analysis (EDA)

The purpose of EDA, in reference to Machine Learning, is to understand the data and its underlying structure, identify patterns, detect anomalies, and gain insights that can inform the choice of machine learning algorithms and feature engineering strategies. EDA helps in making informed decisions about how to preprocess the data, select relevant features, and choose appropriate models for training.

In [ ]:
df_train = pd.DataFrame(X_train, columns=df_clf.columns[:-1])
df_train["target"] = y_train

sns.pairplot(df_train, hue="target", palette="Set1")
plt.suptitle("Pairplot of Iris Dataset", y=1.02)
plt.show()

#### Create the Pipeline

A pipeline is a way to streamline a machine learning workflow by chaining together multiple steps, such as data preprocessing and model training, into a single object. This allows for easier management of the workflow and ensures that the same transformations are applied consistently during both training and testing.

In [ ]:
knn_clf_pipeline = Pipeline(
    [
        # ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier()),
    ]
)

#### Creating the Hyperparameter Grid

A hyperparameter is a setting chosen before training begins. It is not learned from the data; instead, it controls how the learning algorithm behaves during training and prediction.

For specific details about KNN Classifier hyperparameters, please refer to the [scikit-learn documentation](https://scikit-learn.org/1.6/modules/generated/sklearn.neighbors.KNeighborsClassifier.html).

In [ ]:
clf_param_grid = {"knn__n_neighbors": np.arange(1, 15)}

#### Cross-Validation and Model Hyperparameter Tuning

Cross-validation is a technique used to evaluate the performance of a machine learning model and to tune its hyperparameters. It involves partitioning the dataset into multiple subsets, training the model on some subsets while testing it on others, and then averaging the results to get a more reliable estimate of the model's performance.

**Stratified K-Fold Cross-Validation** is a specific type of cross-validation that ensures that each fold of the dataset has the same proportion of classes as the original dataset. This is particularly important for imbalanced datasets, where some classes may be underrepresented. By maintaining the class distribution in each fold, stratified K-Fold Cross-Validation provides a more accurate assessment of the model's performance across all classes.

<fig>
    <img src="https://cdn.jsdelivr.net/gh/ultralytics/assets@main/docs/k-fold-cross-validation-overview.avif">
    <figcaption>Stratified K-Fold Cross-Validation</figcaption>
</fig>

**Grid Search** is a method for systematically working through multiple combinations of hyperparameter values, cross-validating as it goes to determine which combination gives the best performance. It is a brute-force approach to hyperparameter tuning that exhaustively searches through a specified subset of the hyperparameter space.

In [ ]:
cv_clf = StratifiedKFold(n_splits=5,
                         shuffle=True,
                         random_state=SEED)

grid_clf = GridSearchCV(
    estimator=knn_clf_pipeline,
    param_grid=clf_param_grid,
    cv=cv_clf,
    scoring="accuracy",
    n_jobs=-1,
)

#### Fitting the Model to the Training Data

In [ ]:
grid_clf.fit(X_train, y_train)

In [ ]:
print(f"Best Hyperparameters: {grid_clf.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_clf.best_score_:.4f}")

#### Evaluate the KNN Classifier on the Test Set

In [ ]:
# Let's evaluate the model on the test set
test_accuracy_clf = grid_clf.score(X_test, y_test)
print(f"Test Set Accuracy: {test_accuracy_clf:.4f}")

#### Confusion Matrix

In [ ]:
y_pred = grid_clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix Heatmap")
plt.show()

#### Analyzing and Visualizing KNN Classifier

In [ ]:
k_values = range(1, 16)
train_means, train_stds = [], []
test_means, test_stds = [], []

for k in k_values:
    model = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=k)),
        ]
    )

    cv_out = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv_clf,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1,
    )

    train_means.append(cv_out["train_score"].mean())
    train_stds.append(cv_out["train_score"].std())
    test_means.append(cv_out["test_score"].mean())
    test_stds.append(cv_out["test_score"].std())

plt.figure(figsize=(10, 6))

plt.errorbar(
    k_values,
    train_means,
    yerr=train_stds,
    fmt="-o",
    capsize=4,
    label="Train Accuracy",
)

plt.errorbar(
    k_values,
    test_means,
    yerr=test_stds,
    fmt="-s",
    capsize=4,
    label="Validation Accuracy",
)

plt.xticks(list(k_values))
plt.xlabel("k (n_neighbors)")
plt.ylabel("Accuracy")
plt.title("KNN Train/Validation Accuracy vs k using Stratified K-Fold CV")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

#### Using the Pipeline for Inference

In [ ]:
new_iris_data_point = np.array([[5.1, 3.5, 1.4, 0.2]])  # Example data point
predicted_class = grid_clf.predict(new_iris_data_point)
print(
    f"Predicted class for the new data point {new_iris_data_point[0]}: {predicted_class[0]}"
)

In [ ]:
# reset all Jupyter variables
%reset -f

## KNN Regression

The KNN regression algorithm works by finding the K nearest neighbors to a given data point and then averaging their target variable values to make a prediction for that data point.

<fig>
    <img src="https://miro.medium.com/v2/resize:fit:720/format:webp/1*Wf4MVv940JCYIySw_ZyDgQ.png">
    <figcaption>KNN Regression for k=3</figcaption>
</fig>

### Example

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold,
    cross_validate,
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.compose import ColumnTransformer

SEED = 42

#### Load Dataset

In [ ]:
df_reg = load_diabetes(as_frame=True, scaled=False).frame

df_reg.head()

Data Dictionary:

The Diabetes Dataset include following attributes:

| Attribute | Description | Datatype |
| --- | --- | --- |
| age | Age in years | numerical |
| sex | Gender of the patient | categorical |
| bmi | Body mass index | numerical |
| bp | Average blood pressure | numerical |
| s1 | Total serum cholesterol (tc) | numerical |
| s2 | Low-density lipoproteins (ldl) | numerical |
| s3 | High-density lipoproteins (hdl) | numerical |
| s4 | Total cholesterol / HDL (tch) | numerical |
| s5 | Possibly log of serum triglycerides level (ltg) | numerical |
| s6 | Blood sugar level (glu) | numerical |
| target | A quantitative measure of disease progression one year after baseline | numerical |

#### Inspect Dataset

In [ ]:
df_reg.info()

In [ ]:
df_reg.describe()

**Setting up a Baseline using the Mean**

From the data, we can calculate the mean and standard deviation of the target variable and use them as a baseline for our regression model. This means that if our KNN regression model cannot outperform the mean in terms of prediction accuracy, then it may not be a useful model for our data. This can be checked using the $R^2$ metric.

Quick rule with $R^2$:

1. $R^2=0$ means you are doing exactly as well as predicting the mean.
2. $R^2<0$ means you are worse than that baseline.
3. $R^2>0$ means you beat the baseline.

In [ ]:
# Mean Calculation
mean_target = df_reg["target"].mean()
print(f"Mean of Target Variable: {mean_target:.4f}")

#### Train-Test Split (75-25 split)

In [ ]:
X = df_reg.drop("target", axis=1)
y = df_reg["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

#### Exploratory Data Analysis (EDA)

In [ ]:
categorical_column = ["sex"]
numerical_columns = df_reg.columns.drop(["target", "sex"]).tolist()

df_train = pd.DataFrame(X_train, columns=df_reg.columns[:-1])
df_train["target"] = y_train

# plot the histograms in subplots
df_train[numerical_columns].hist(bins=20, figsize=(15, 10))
plt.suptitle("Histograms of Diabetes Dataset Features and Target")
plt.show()

#### Create the Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    [("scaler", StandardScaler(), numerical_columns)],
    remainder="passthrough", # keep the categorical column as is
)

knn_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("knn", KNeighborsRegressor())
    ]
)

**Difference between Pipeline versus ColumnTransformer?**

A `Pipeline` is used when you want to apply a sequence of transformations and a final estimator to your data. It is useful for chaining together multiple steps, such as preprocessing and model training, into a single object. This ensures that the same transformations are applied consistently during both training and testing.

A `ColumnTransformer` is used when you want to apply different transformations to different columns of your data. It is useful for handling mixed data types or applying specific preprocessing steps to subsets of features.

Together, these tools provide a flexible framework for preprocessing and modeling data in scikit-learn.

#### Creating the Hyperparameter Grid

For specific details about KNN Regression hyperparameters, please refer to the [scikit-learn documentation](https://scikit-learn.org/1.6/modules/generated/sklearn.neighbors.KNeighborsRegressor.html).

In [ ]:
param_grid = {"knn__n_neighbors": np.arange(1, 20)}

#### Cross-Validation and Model Hyperparameter Tuning

For regression tasks KFold Cross-Validation is used instead of Stratified K-Fold Cross-Validation, as the latter is specifically designed for classification tasks where the target variable is categorical.

In regression tasks, the target variable is continuous, and there is no need to maintain class proportions across folds. Therefore, standard K-Fold Cross-Validation is more appropriate for evaluating regression models like KNN regression.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
)

#### Fitting the Model to the Training Data

In [ ]:
grid.fit(X_train, y_train)

In [ ]:
print(f"Best Hyperparameters: {grid.best_params_}")
print(f"Best R² Score: {grid.best_score_:.4f}")

#### Evaluate the KNN Classifier on the Test Set

In [ ]:
# Let's evaluate the model on the test set
test_r2 = grid.score(X_test, y_test)
print(f"Test Set R²: {test_r2:.4f}")

# Using RMSE as an additional metric for evaluation (use numpy)
test_rmse = np.sqrt(np.mean((y_test - grid.predict(X_test)) ** 2))
print(f"Test Set RMSE: {test_rmse:.4f}")

In [ ]:
df_reg.describe()


#### Analyzing and Visualizing KNN Regressor

In [ ]:
k_values = range(1, 20)
train_means, train_stds = [], []
val_means, val_stds = [], []

for k in k_values:
    model = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("knn", KNeighborsRegressor(n_neighbors=k)),
        ]
    )

    cv_out = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="r2",
        return_train_score=True,
        n_jobs=-1,
    )

    train_means.append(cv_out["train_score"].mean())
    train_stds.append(cv_out["train_score"].std())
    val_means.append(cv_out["test_score"].mean())
    val_stds.append(cv_out["test_score"].std())

best_idx = int(np.argmax(val_means))
best_k = list(k_values)[best_idx]
best_val = val_means[best_idx]

plt.figure(figsize=(10, 6))
plt.errorbar(
    k_values,
    train_means,
    yerr=train_stds,
    fmt="-o",
    capsize=4,
    label="Train R²",
)
plt.errorbar(
    k_values,
    val_means,
    yerr=val_stds,
    fmt="-s",
    capsize=4,
    label="Validation R²",
)

plt.xticks(list(k_values))
plt.xlabel("k (n_neighbors)")
plt.ylabel("R²")
plt.ylim(-0.5, 1.01)
plt.title("KNN Regressor Train/Validation R² vs k using K-Fold CV")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print(f"Best k by validation R²: {best_k}")
print(f"Best mean validation R²: {best_val:.4f}")

In [ ]:
# reset all Jupyter variables
%reset -f